In [2]:
import pandas as pd
import numpy as np
import os

# Mapping Setup

In [3]:
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/"

In [7]:
# ── 1. UniProt ────────────────────────────────────────────────────────────────
# Full UniProt ↔ NCBI mapping table (used for cross-referencing protein/gene IDs)
Uniprot_names = pd.read_csv(f'{BASE_PATH}databases_for_mapping/uniprot/UNIPROT_NCBI_ALL_MAPPED.csv')

# UniProt Accession → Recommended Protein Name (RecName) mapping
# RecName is the curated "recommended name" from UniProt's controlled vocabulary
Uniprot_Recname = pd.read_csv(f'{BASE_PATH}databases_for_mapping/uniprot/Uniprot_ID_Recname.csv')
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['AC'], Uniprot_Recname['RecName']))
# e.g. {'P04637': 'Cellular tumor antigen p53', ...}


/tmp/ipykernel_2405180/1746856877.py:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  Uniprot_names = pd.read_csv(f'{BASE_PATH}databases_for_mapping/uniprot/UNIPROT_NCBI_ALL_MAPPED.csv')


In [8]:
# ── 2. Tissue Ontology (BTO) ──────────────────────────────────────────────────
# BTO (BRENDA Tissue Ontology): maps tissue/cell-type names → BTO IDs
# Used to normalize free-text tissue names into ontology-controlled IDs
BTO = pd.read_csv(f'{BASE_PATH}databases_for_mapping/bto/BTO_ALL_COMBINED.csv')
BTO_dict = dict(zip(BTO['name'], BTO['ID']))
# e.g. {'liver': 'BTO:0000759', 'blood plasma': 'BTO:0000131', ...}

In [14]:
# ── 3. PubChem ────────────────────────────────────────────────────────────────

# 3a. CID-Synonym filtered file: maps chemical synonyms/names → PubChem CIDs
#     Columns: [0] = CID (int), [1] = synonym (string)
#     No header in this file, hence header=None
Pubchem_Syn_fil = pd.read_csv(f'{BASE_PATH}databases_for_mapping/pubchem/CID-Synonym-filtered',
    sep='\t',
    header=None
)

# Sanity checks (run interactively to verify mappings are loading correctly)
# Pubchem_Syn_fil[Pubchem_Syn_fil[1] == 'metformin']  # check metformin CID
# Pubchem_Syn_fil[Pubchem_Syn_fil[0] == 4091]          # check CID 4091 synonyms

# synonym → CID lookup dict (lowercased keys for case-insensitive matching)
# Note: if a synonym maps to multiple CIDs, last one wins (dict deduplication)
Pubchem_Syn_fil_dict       = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))          # original case
Pubchem_Syn_fil_dict_lower = dict(zip(Pubchem_Syn_fil[1].str.lower(), Pubchem_Syn_fil[0]))  # lowercase for fuzzy match


# 3b. Combined PubChem compound table: CID → IUPAC name + SMILES
#     Pre-built pickle for fast loading (large file)
Pubchem = pd.read_pickle(f'{BASE_PATH}databases_for_mapping/pubchem/combined_df.pkl')

# CID → canonical SMILES (used for chemical structure representation in KG)
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))

# CID → IUPAC name (used as Head_detail_name for ChemicalEntity nodes)
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))

In [9]:
# ── 4. Ensembl → NCBI Gene Symbol Mapping ────────────────────────────────────
# Raw table: 86,402 Ensembl IDs mapped to NCBI gene symbols
# 'name' = NCBI gene symbol, 'initial_alias' = Ensembl ID
ENS_2NCBI = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')

# Drop rows with no gene symbol — leaves ~42,530 valid mappings
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]

# Gene Symbol → Ensembl ID lookup dict
# Used to annotate NCBI_ALL_GENE with Ensembl IDs below
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))

# Free memory — large file, not needed beyond this mapping
del ENS_2NCBI

In [10]:
# ── 5. Human NCBI Gene Info ───────────────────────────────────────────────────
# Official NCBI gene_info file for Homo sapiens
# Columns of interest: GeneID, Symbol, Synonyms, description
NCBI_ALL_GENE = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ncbi/Homo_sapiens.gene_info',
    sep='\t'
)

# Annotate each gene with its Ensembl ID (via Symbol → Ensembl dict built above)
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)

# GeneID → gene description  (used as Head/Tail_detail_name for Gene nodes)
NCBI_ALL_GENEname_dict    = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['description']))

# GeneID → gene Symbol       (used to convert numeric IDs back to readable symbols)
NCBI_ALL_GENEIDname_dict  = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['Symbol']))

# Gene Symbol → gene description  (direct symbol-based lookup, no ID needed)
NCBI_ALL_Symb_Desc_dict   = dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))

# Synonym → canonical Gene Symbol (pipe-delimited field in NCBI, e.g. "TP53|P53|LFS1")
# Step 1: Build raw dict — keys are the full pipe-joined synonym strings
NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))

# Step 2: Explode pipe-delimited synonyms so each alias gets its own entry
# e.g. "TP53|P53|LFS1" → {'TP53': 'TP53', 'P53': 'TP53', 'LFS1': 'TP53'}
# Used to recover canonical symbols from non-standard gene name variants
exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for k, v in NCBI_ALL_GENE_Syn_Dict.items():
    for single_key in k.split('|'):
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[single_key.strip()] = v

In [11]:

# ── 6. Mouse NCBI Gene Info ───────────────────────────────────────────────────
# Official NCBI gene_info file for Mus musculus
NCBI_MOUSE_gene = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ncbi/Mus_musculus.gene_info',
    sep='\t'
)

# Extract Ensembl Mouse Gene ID (ENSMUSG...) from the pipe-delimited dbXrefs column
# e.g. 'Ensembl:ENSMUSG00000051951|...' → 'ENSMUSG00000051951'
NCBI_MOUSE_gene['ENSMBL'] = NCBI_MOUSE_gene['dbXrefs'].str.extract(r'(ENSMUSG\d+)', expand=False)

# Ensembl Mouse ID → Mouse Gene Symbol  (for Ensembl-based input data)
NCBI_Mouse_gene_Locus_2_GeneSymd_ict = dict(zip(NCBI_MOUSE_gene['ENSMBL'],  NCBI_MOUSE_gene['Symbol']))

# Mouse Gene Symbol → description       (used as Head/Tail_detail_name for Mouse Gene nodes)
NCBI_Mouse_SYM_Descrip_dict = dict(zip(NCBI_MOUSE_gene['Symbol'], NCBI_MOUSE_gene['description']))


# ── 7. Gene Ontology (GO) ─────────────────────────────────────────────────────
# GO term table: GO ID, human-readable name, and namespace (domain)
GO = pd.read_csv(f'{BASE_PATH}databases_for_mapping/MESH/MESH_GO_ID_NAME.csv')

# GO ID → term name  (e.g. 'GO:0008150' → 'biological_process')
GO_dict = dict(zip(GO['id'], GO['name']))

# Normalize namespace labels to title case for display consistency in the KG
GO["namespace"] = GO["namespace"].replace({
    "biological_process": "Biological Process",
    "molecular_function": "Molecular Function",
    "cellular_component": "Cellular Component"
})

# GO ID → namespace  (used to tag GO nodes with their domain/type)
GO_namespace_dict = dict(zip(GO['id'], GO['namespace']))

In [28]:
#

# AnnoMO data

# 8

In [147]:
Anti_aging_intervention = pd.read_excel(f'{BASE_PATH}ageannomo/AgeAnnoMO-main/Anti-aging interventions/Intervention.xlsx')
Anti_aging_intervention

,Species,Data type,Hallmarks,Anti-aging interventions,Targets,Effect,Publication,2015,Pubmed,Link
0,mouse,"tissue data, cell line",Genomic instability,SIRT2 activation and NAD,BIBR1,increases median lifespan,SIRT2 induces the checkpoint kinase BubR1 to i...,2014,24825348.0,https://pubmed.ncbi.nlm.nih.gov/24825348/
1,long-lived species,tissue data,Genomic instability,SIRT6 activation,DNA double-strand break,increases median lifespan,SIRT6 Is Responsible for More Efficient DNA Do...,2019,31002797.0,https://pubmed.ncbi.nlm.nih.gov/31002797/
2,"human, mouse",tissue data,Genomic instability,farnesyl transferase inhibitors,LMNA,lifespan extension,Progeria: a paradigm for translational medicine,2013,24485450.0,https://pubmed.ncbi.nlm.nih.gov/24485450/
3,NaN,cell line,Genomic instability,Kinase Inhibitors,JAK/STAT pathway,enhance healthspan and increase longevity,Application of Kinase Inhibitors for Anti-agin...,2017,28714416.0,https://pubmed.ncbi.nlm.nih.gov/28714416/
4,human,tissue data,Genomic instability,antioxidants,ROS,anti UV exposure,Role of antioxidants in the skin: Anti-aging e...,2010,20399614.0,https://pubmed.ncbi.nlm.nih.gov/20399614/
...,...,...,...,...,...,...,...,...,...,...
86,mouse,tissue data,Dysbiosis,Lactobacillus plantarum,amyloid-β protein,promotes longevity,Lactobacillus plantarum GKM3 Promotes Longevit...,2021,34445020.0,https://pubmed.ncbi.nlm.nih.gov/34445020/
87,mouse,tissue data,Dysbiosis,Fecal microbiota transplantation,NaN,restoration of intestinal microflora functions,Functional Restoration of Bacteriomes and Viro...,2021,33577875.0,https://pubmed.ncbi.nlm.nih.gov/33577875/
88,mouse,tissue data,Dysbiosis,postbiotics,NaN,benefit aged people’s health,Probiotics and their Metabolites Reduce Oxidat...,2022,35157139.0,https://pubmed.ncbi.nlm.nih.gov/35157139/
89,NaN,cell line,Dysbiosis,metformin,"Complex I, AMPK, and glycerophosphate dehydrog...",pro-longevity properties,Pleiotropic effects of metformin: Shaping the ...,2018,30336272.0,https://pubmed.ncbi.nlm.nih.gov/30336272/


In [148]:
print(set(Anti_aging_intervention['Species']))
print(Anti_aging_intervention['Species'].value_counts())

{'human', 'Zebrafish', 'long-lived species', 'Human', 'Human Erythrocyte', 'rat', 'nonhuman primates', 'mouse, monkey', 'Rat', 'rhesus monkeys', 'Caenorhabditis elegans', 'human, mouse', 'mouse', 'Drosophila melanogaster', nan}
Species
mouse                      45
human                      17
Rat                         4
rat                         3
Drosophila melanogaster     3
Zebrafish                   3
Caenorhabditis elegans      2
mouse, monkey               1
long-lived species          1
human, mouse                1
rhesus monkeys              1
Human Erythrocyte           1
nonhuman primates           1
Human                       1
Name: count, dtype: int64


In [149]:
# Mapping aging hallmarks to GO IDs.
# These GO IDs were checked in QuickGO — most hallmarks did not have official GO terms available,
# so placeholder/custom IDs (GO:111111x) were assigned internally.
# Exception: "Cellular senescence" has a real, verified GO term (GO:0090398) from QuickGO.
    
aging_hallmarks_to_go = {
    "Genomic instability": "GO:1111111",
    "Telomere attrition": "GO:1111112",
    "Epigenetic alterations": "GO:1111113",
    "Epigenetic alteration": "GO:1111113",
    "Loss of proteostasis": "GO:1111114",
    "Deregulated nutrient-sensing": "GO:1111115",
    "Mitochondrial dysfunction": "GO:1111116",
    "Cellular senescence": "GO:0090398",
    "Stem cell exhaustion": "GO:1111117",
    "Altered intercellular communication": "GO:1111118"
}

## 
Anti_aging_intervention = pd.read_excel(f'{BASE_PATH}ageannomo/AgeAnnoMO-main/Anti-aging interventions/Intervention.xlsx')
print(set(Anti_aging_intervention['Species']))


# Inspect raw species values before standardization
print(set(Anti_aging_intervention['Species']))


# ─────────────────────────────────────────────────────────────────
# STANDARDIZE Species names to full scientific names
# Multi-species entries (e.g. 'human, mouse') are mapped to a
# placeholder label and will be dropped by the Sp_list filter below
# ─────────────────────────────────────────────────────────────────
Anti_aging_intervention.loc[Anti_aging_intervention['Species'] == 'human',         'Species'] = 'Homo sapiens'
Anti_aging_intervention.loc[Anti_aging_intervention['Species'] == 'mouse',         'Species'] = 'Mus musculus'
Anti_aging_intervention.loc[Anti_aging_intervention['Species'] == 'human, mouse',  'Species'] = 'Human'      # Will be dropped below
Anti_aging_intervention.loc[Anti_aging_intervention['Species'] == 'mouse, monkey', 'Species'] = 'Mouse'      # Will be dropped below
Anti_aging_intervention.loc[Anti_aging_intervention['Species'] == 'Zebrafish',     'Species'] = 'Danio rerio'

# Keep only the five target species; drops multi-species placeholder rows
Sp_list = ['Caenorhabditis elegans', 'Drosophila melanogaster',
           'Homo sapiens', 'Mus musculus', 'Danio rerio']
Anti_aging_intervention = Anti_aging_intervention[Anti_aging_intervention['Species'].isin(Sp_list)]
Anti_aging_intervention


# ─────────────────────────────────────────────────────────────────
# RENAME COLUMNS to match KG schema
# ─────────────────────────────────────────────────────────────────
Anti_aging_intervention.rename(columns={
    'Hallmarks': 'Head',              # Aging hallmark → Head node
    'Targets':   'Tail',              # Gene/protein target → Tail node
    'Pubmed':    'Their_Source_PMID'  # PubMed citation
}, inplace=True)

# Reclassify 'Disabled macroautophagy' under the standard hallmark name
# so it maps correctly to GO:1111114 via aging_hallmarks_to_go
Anti_aging_intervention.loc[
    Anti_aging_intervention['Head'] == 'Disabled macroautophagy', 'Head'
] = 'Loss of proteostasis'

Anti_aging_intervention

{'human', 'Zebrafish', 'long-lived species', 'Human', 'Human Erythrocyte', 'rat', 'nonhuman primates', 'mouse, monkey', 'Rat', 'rhesus monkeys', 'Caenorhabditis elegans', 'human, mouse', 'mouse', 'Drosophila melanogaster', nan}
{'human', 'Zebrafish', 'long-lived species', 'Human', 'Human Erythrocyte', 'rat', 'nonhuman primates', 'mouse, monkey', 'Rat', 'rhesus monkeys', 'Caenorhabditis elegans', 'human, mouse', 'mouse', 'Drosophila melanogaster', nan}


,Species,Data type,Head,Anti-aging interventions,Tail,Effect,Publication,2015,Their_Source_PMID,Link
0,Mus musculus,"tissue data, cell line",Genomic instability,SIRT2 activation and NAD,BIBR1,increases median lifespan,SIRT2 induces the checkpoint kinase BubR1 to i...,2014,24825348.0,https://pubmed.ncbi.nlm.nih.gov/24825348/
4,Homo sapiens,tissue data,Genomic instability,antioxidants,ROS,anti UV exposure,Role of antioxidants in the skin: Anti-aging e...,2010,20399614.0,https://pubmed.ncbi.nlm.nih.gov/20399614/
5,Mus musculus,tissue data,Genomic instability,Arf/p53 activation,Arf/p53,alleviating the load of age-associated damage,Delayed ageing through damage protection by th...,2007,17637672.0,https://pubmed.ncbi.nlm.nih.gov/17637672/
7,Homo sapiens,tissue data,Telomere attrition,Centella asiatica extract formulation,NaN,Telomere activation,Discovery of potent telomerase activators: Unf...,2019,31485647.0,https://pubmed.ncbi.nlm.nih.gov/31485647/
8,Homo sapiens,tissue data,Telomere attrition,Astragalus extract formulation,NaN,Telomere activation,Discovery of potent telomerase activators: Unf...,2019,31485647.0,https://pubmed.ncbi.nlm.nih.gov/31485647/
...,...,...,...,...,...,...,...,...,...,...
85,Mus musculus,tissue data,Dysbiosis,fecal microbiota transplantation,NaN,against age-related diseases,Healthspan and lifespan extension by fecal mic...,2019,31332389.0,https://pubmed.ncbi.nlm.nih.gov/31332389/
86,Mus musculus,tissue data,Dysbiosis,Lactobacillus plantarum,amyloid-β protein,promotes longevity,Lactobacillus plantarum GKM3 Promotes Longevit...,2021,34445020.0,https://pubmed.ncbi.nlm.nih.gov/34445020/
87,Mus musculus,tissue data,Dysbiosis,Fecal microbiota transplantation,NaN,restoration of intestinal microflora functions,Functional Restoration of Bacteriomes and Viro...,2021,33577875.0,https://pubmed.ncbi.nlm.nih.gov/33577875/
88,Mus musculus,tissue data,Dysbiosis,postbiotics,NaN,benefit aged people’s health,Probiotics and their Metabolites Reduce Oxidat...,2022,35157139.0,https://pubmed.ncbi.nlm.nih.gov/35157139/


In [150]:
# Inspect species distribution after filtering
Anti_aging_intervention['Species'].value_counts()


# ─────────────────────────────────────────────────────────────────
# SPLIT by Species for separate per-organism processing
# ─────────────────────────────────────────────────────────────────
Human_df_file = Anti_aging_intervention[Anti_aging_intervention['Species'] == 'Homo sapiens']
Mouse_df_file  = Anti_aging_intervention[Anti_aging_intervention['Species'] == 'Mus musculus']
Droso_df_file  = Anti_aging_intervention[Anti_aging_intervention['Species'] == 'Drosophila melanogaster']
Cele_df_file   = Anti_aging_intervention[Anti_aging_intervention['Species'] == 'Caenorhabditis elegans']
Zebra_df_file  = Anti_aging_intervention[Anti_aging_intervention['Species'] == 'Danio rerio']


In [151]:


# ═════════════════════════════════════════════════════════════════
# FILE 4: HUMAN — Chemical → Aging Hallmark (BiologicalProcess)
# Direction after swap: PubChem CID (chemical) → GO ID (hallmark)
# ═════════════════════════════════════════════════════════════════

# Step 1: Drop rows with no intervention compound listed
Human_df_file = Human_df_file[~Human_df_file['Anti-aging interventions'].isna()]

# Step 2: Lowercase intervention names for consistent dictionary lookup
Human_df_file['Anti-aging interventions'] = (
    Human_df_file['Anti-aging interventions'].astype(str).str.lower()
)

# Step 3: Map aging hallmark names → GO IDs (see aging_hallmarks_to_go above)
Human_df_file['Head'] = Human_df_file['Head'].map(aging_hallmarks_to_go)

# Step 4: Drop rows where hallmark had no GO mapping
Human_df_file = Human_df_file[~Human_df_file['Head'].isna()]

# Step 5: Map lowercased intervention name → PubChem CID via synonym dictionary
Human_df_file['Tail'] = Human_df_file['Anti-aging interventions'].map(Pubchem_Syn_fil_dict)

# Step 6: Remove trailing '.0' from CID values (artifact of float conversion)
Human_df_file["Tail"] = Human_df_file["Tail"].astype(str).str.replace(r'\.0$', '', regex=True)

# Step 7: Map PubChem CID → IUPAC name for human-readable chemical identity
Human_df_file['Tail_detail_name'] = Human_df_file['Tail'].map(Pubchem_IUPAC_CID_Dict)
Human_df_file


# ─────────────────────────────────────────────────────────────────
# RETRY MAPPING: Capitalize and re-attempt for unresolved compounds
# Some compounds fail lowercase lookup but succeed when capitalized
# ─────────────────────────────────────────────────────────────────

# Identify rows where CID→IUPAC mapping failed
mask = Human_df_file['Tail_detail_name'].isna()

# Capitalize intervention name and retry synonym lookup
Human_df_file.loc[mask, 'Anti-aging interventions'] = (
    Human_df_file.loc[mask, 'Anti-aging interventions']
    .astype(str).str.strip().str.capitalize()
)

# Re-map capitalized name → PubChem CID
Human_df_file.loc[mask, 'Tail'] = (
    Human_df_file.loc[mask, 'Anti-aging interventions'].map(Pubchem_Syn_fil_dict)
)

# Remove trailing '.0' again after remapping
Human_df_file["Tail"] = Human_df_file["Tail"].astype(str).str.replace(r'\.0$', '', regex=True)

# Re-attempt IUPAC name lookup for previously unresolved rows
Human_df_file.loc[mask, 'Tail_detail_name'] = (
    Human_df_file.loc[mask, 'Tail'].map(Pubchem_IUPAC_CID_Dict)
)
Human_df_file

# Drop rows that still have no resolved chemical (compound not in PubChem)
Human_df_file = Human_df_file[~Human_df_file['Tail_detail_name'].isna()]
Human_df_file


# ─────────────────────────────────────────────────────────────────
# SWAP Head/Tail → final edge direction: Chemical → BiologicalProcess
# Before: Head = GO ID (hallmark), Tail = PubChem CID (chemical)
# After:  Head = PubChem CID,      Tail = GO ID (hallmark)
# ─────────────────────────────────────────────────────────────────
Human_df_file[['Head', 'Tail']] = Human_df_file[['Tail', 'Head']]

# Rename Tail_detail_name → Head_Detail_name (now describes the Head/chemical)
Human_df_file = Human_df_file.rename(columns={'Tail_detail_name': 'Head_Detail_name'})
Human_df_file


# ─────────────────────────────────────────────────────────────────
# REVERSE LOOKUP: GO ID → Hallmark name (Tail detail after swap)
# Inverse of aging_hallmarks_to_go defined above
# ─────────────────────────────────────────────────────────────────
hallmark_to_go = {
    "GO:1111111": "Genomic Instability",
    "GO:1111112": "Telomere Attrition",
    "GO:1111113": "Epigenetic Alterations",
    "GO:1111114": "Loss of Proteostasis",
    "GO:1111115": "Deregulated Nutrient Sensing",
    "GO:1111116": "Mitochondrial Dysfunction",
    "GO:0090398": "Cellular Senescence",            # Official QuickGO term
    "GO:1111117": "Stem Cell Exhaustion",
    "GO:1111118": "Altered Intercellular Communication"
}

# Map GO ID (now in Tail) → human-readable hallmark name
# NOTE: columns are still mixed-case here; 'tail' (lowercase) will produce
# all NaNs until .str.lower() is applied below — use 'Tail' instead

Human_df_file

,Species,Data type,Head,Anti-aging interventions,Tail,Effect,Publication,2015,Their_Source_PMID,Link,Head_Detail_name
11,Homo sapiens,tissue data,73659,Maslinic acid,GO:1111112,Telomere activation,Discovery of potent telomerase activators: Unf...,2019,31485647.0,https://pubmed.ncbi.nlm.nih.gov/31485647/,"(4aS,6aR,6aS,6bR,8aR,10R,11R,12aR,14bS)-10,11-..."
39,Homo sapiens,tissue data,5702063,guanabenz,GO:1111114,protective effect on disease-related multiple ...,Guanabenz mitigates the neuropathological alte...,2022,35195784.0,https://pubmed.ncbi.nlm.nih.gov/35195784/,"2-[(E)-(2,6-dichlorophenyl)methylideneamino]gu..."
49,Homo sapiens,tissue data,1102,spermidine,GO:1111114,extends lifespan across species,Spermidine: a physiological autophagy inducer ...,2018,30306826.0,https://pubmed.ncbi.nlm.nih.gov/30306826/,"N'-(3-aminopropyl)butane-1,4-diamine"
57,Homo sapiens,tissue data,11764719,Elamipretide,GO:1111116,improvement in BTHS symptoms,A phase 2/3 randomized clinical trial followed...,2021,33077895.0,https://pubmed.ncbi.nlm.nih.gov/33077895/,(2S)-6-amino-2-[[(2S)-2-[[(2R)-2-amino-5-(diam...
63,Homo sapiens,tissue data,4091,metformin,GO:1111115,the significant effect of metformin on cogniti...,Effect of Metformin on Adult Hippocampal Neuro...,2017,28378260.0,https://pubmed.ncbi.nlm.nih.gov/28378260/,"3-(diaminomethylidene)-1,1-dimethylguanidine"


In [152]:
# all NaNs until .str.lower() is applied below — use 'Tail' instead
Human_df_file['tail_detail_name'] = Human_df_file['Tail'].map(hallmark_to_go)

In [153]:
# ─────────────────────────────────────────────────────────────────
# ADD METADATA COLUMNS for KG schema
# ─────────────────────────────────────────────────────────────────
Human_df_file['head_type']       = 'ChemicalEntity'
Human_df_file['tail_type']       = 'BiologicalProcess'
Human_df_file['Relation']        = 'ChemicalEntity_BiologicalProcess'
Human_df_file['head_id_is']      = 'Pubchem'    # ID namespace for Head node
Human_df_file['tail_id_is']      = 'Quick_GO'   # ID namespace for Tail node
Human_df_file['relation_source'] = 'AgeAnnoMO'  # Source database
Human_df_file


# ─────────────────────────────────────────────────────────────────
# FINAL CLEANUP
# ─────────────────────────────────────────────────────────────────

# Standardize all column names to lowercase for consistency across KG files
Human_df_file.columns = Human_df_file.columns.str.lower()
Human_df_file

# Drop any duplicate columns (can arise from repeated renames/merges)
Human_df_file = Human_df_file.loc[:, ~Human_df_file.columns.duplicated()]
Human_df_file
Human_df_file.to_csv(f'{OUT_PATH}ageanno_processed_mo/Human_Chemical_BiologicalProcess_1.csv',index = None)

In [154]:
# ─────────────────────────────────────────────────────────────────
# FILE 5: MOUSE — Chemical → Aging Hallmark (BiologicalProcess)
# Same pipeline as Human (File 4) but for Mus musculus
# ─────────────────────────────────────────────────────────────────

# Step 1: Drop rows with no intervention compound listed
Mouse_df_file = Mouse_df_file[~Mouse_df_file['Anti-aging interventions'].isna()]

# Step 2: Lowercase intervention names for consistent dictionary lookup
Mouse_df_file['Anti-aging interventions'] = (
    Mouse_df_file['Anti-aging interventions'].astype(str).str.lower()
)

# Step 3: Map aging hallmark names → GO IDs (custom + official; see aging_hallmarks_to_go)
Mouse_df_file['Head'] = Mouse_df_file['Head'].map(aging_hallmarks_to_go)

# BUG: This line assigns to Human_df_file instead of Mouse_df_file — should be:
# Mouse_df_file = Mouse_df_file[~Mouse_df_file['Head'].isna()]
Human_df_file = Mouse_df_file[~Mouse_df_file['Head'].isna()]

# Step 4: Map lowercased intervention name → PubChem CID using synonym dictionary
Mouse_df_file['Tail'] = Mouse_df_file['Anti-aging interventions'].map(Pubchem_Syn_fil_dict)

# Step 5: Remove trailing '.0' from CID values (artifact of float conversion)
Mouse_df_file["Tail"] = Mouse_df_file["Tail"].astype(str).str.replace(r'\.0$', '', regex=True)

# Step 6: Map PubChem CID → IUPAC name for human-readable chemical identity
Mouse_df_file['Tail_detail_name'] = Mouse_df_file['Tail'].map(Pubchem_IUPAC_CID_Dict)
Mouse_df_file


,Species,Data type,Head,Anti-aging interventions,Tail,Effect,Publication,2015,Their_Source_PMID,Link,Tail_detail_name
0,Mus musculus,"tissue data, cell line",GO:1111111,sirt2 activation and nad,nan,increases median lifespan,SIRT2 induces the checkpoint kinase BubR1 to i...,2014,24825348.0,https://pubmed.ncbi.nlm.nih.gov/24825348/,NaN
5,Mus musculus,tissue data,GO:1111111,arf/p53 activation,nan,alleviating the load of age-associated damage,Delayed ageing through damage protection by th...,2007,17637672.0,https://pubmed.ncbi.nlm.nih.gov/17637672/,NaN
16,Mus musculus,tissue data,GO:1111112,telomerase reverse transcriptase,nan,antiaging activity,Telomerase Reverse Transcriptase Delays Aging ...,2008,19013273.0,https://pubmed.ncbi.nlm.nih.gov/19013273/,NaN
17,Mus musculus,tissue data,GO:1111112,telomerase reactivation,nan,lifespan extension,Telomerase reactivation reverses tissue degene...,2011,21113150.0,https://pubmed.ncbi.nlm.nih.gov/21113150/,NaN
18,Mus musculus,tissue data,GO:1111112,telomerase gene therapy,nan,delaying physiological aging and extending lon...,Telomerase gene therapy in adult and old mice ...,2012,22585399.0,https://pubmed.ncbi.nlm.nih.gov/22585399/,NaN
19,Mus musculus,tissue data,GO:1111112,hyperlong telomeres,nan,Hyper-long telomere mice also have less incide...,Mice with hyper-long telomeres show less metab...,2019,31624261.0,https://pubmed.ncbi.nlm.nih.gov/31624261/,NaN
20,Mus musculus,tissue data,GO:1111112,tert-directed transcriptional activities,nan,attenuating AD progression including cognitive...,Telomerase reverse transcriptase preserves neu...,2021,35036927.0,https://pubmed.ncbi.nlm.nih.gov/35036927/,NaN
21,Mus musculus,tissue data,GO:1111112,telomerase activation by gene therapy strategy,nan,telomerase activation may represent an effecti...,Therapeutic effects of telomerase in mice with...,2018,29378675.0,https://pubmed.ncbi.nlm.nih.gov/29378675/,NaN
22,Mus musculus,tissue data,GO:1111112,telomerase activation by gene therapy strategy,nan,telomerase expression leads to telomere elonga...,Telomerase gene therapy rescues telomere lengt...,2016,26903545.0,https://pubmed.ncbi.nlm.nih.gov/26903545/,NaN
24,Mus musculus,tissue data,GO:1111113,caloric restriction,nan,help to healthy aging and longevity,Diverse interventions that extend mouse lifesp...,2017,28351383.0,https://pubmed.ncbi.nlm.nih.gov/28351383/,NaN


In [155]:
# ─────────────────────────────────────────────────────────────────
# RETRY MAPPING: Capitalize and re-attempt for unresolved compounds
# Some compounds fail lowercase lookup but succeed when capitalized
# ─────────────────────────────────────────────────────────────────

# Step 1: Identify rows where CID→IUPAC mapping failed (Tail_detail_name is NaN)
mask = Mouse_df_file['Tail_detail_name'].isna()  # Note: comment in original mentioning Mesh_supp_PUBCHEM_dict appears to be a leftover — mask is based on Tail_detail_name

# Step 2: Capitalize the intervention name and retry synonym lookup
Mouse_df_file.loc[mask, 'Anti-aging interventions'] = (
    Mouse_df_file.loc[mask, 'Anti-aging interventions']
    .astype(str)
    .str.strip()
    .str.capitalize()
)
Mouse_df_file

,Species,Data type,Head,Anti-aging interventions,Tail,Effect,Publication,2015,Their_Source_PMID,Link,Tail_detail_name
0,Mus musculus,"tissue data, cell line",GO:1111111,Sirt2 activation and nad,nan,increases median lifespan,SIRT2 induces the checkpoint kinase BubR1 to i...,2014,24825348.0,https://pubmed.ncbi.nlm.nih.gov/24825348/,NaN
5,Mus musculus,tissue data,GO:1111111,Arf/p53 activation,nan,alleviating the load of age-associated damage,Delayed ageing through damage protection by th...,2007,17637672.0,https://pubmed.ncbi.nlm.nih.gov/17637672/,NaN
16,Mus musculus,tissue data,GO:1111112,Telomerase reverse transcriptase,nan,antiaging activity,Telomerase Reverse Transcriptase Delays Aging ...,2008,19013273.0,https://pubmed.ncbi.nlm.nih.gov/19013273/,NaN
17,Mus musculus,tissue data,GO:1111112,Telomerase reactivation,nan,lifespan extension,Telomerase reactivation reverses tissue degene...,2011,21113150.0,https://pubmed.ncbi.nlm.nih.gov/21113150/,NaN
18,Mus musculus,tissue data,GO:1111112,Telomerase gene therapy,nan,delaying physiological aging and extending lon...,Telomerase gene therapy in adult and old mice ...,2012,22585399.0,https://pubmed.ncbi.nlm.nih.gov/22585399/,NaN
19,Mus musculus,tissue data,GO:1111112,Hyperlong telomeres,nan,Hyper-long telomere mice also have less incide...,Mice with hyper-long telomeres show less metab...,2019,31624261.0,https://pubmed.ncbi.nlm.nih.gov/31624261/,NaN
20,Mus musculus,tissue data,GO:1111112,Tert-directed transcriptional activities,nan,attenuating AD progression including cognitive...,Telomerase reverse transcriptase preserves neu...,2021,35036927.0,https://pubmed.ncbi.nlm.nih.gov/35036927/,NaN
21,Mus musculus,tissue data,GO:1111112,Telomerase activation by gene therapy strategy,nan,telomerase activation may represent an effecti...,Therapeutic effects of telomerase in mice with...,2018,29378675.0,https://pubmed.ncbi.nlm.nih.gov/29378675/,NaN
22,Mus musculus,tissue data,GO:1111112,Telomerase activation by gene therapy strategy,nan,telomerase expression leads to telomere elonga...,Telomerase gene therapy rescues telomere lengt...,2016,26903545.0,https://pubmed.ncbi.nlm.nih.gov/26903545/,NaN
24,Mus musculus,tissue data,GO:1111113,Caloric restriction,nan,help to healthy aging and longevity,Diverse interventions that extend mouse lifesp...,2017,28351383.0,https://pubmed.ncbi.nlm.nih.gov/28351383/,NaN


In [156]:
#

In [157]:

# Step 3: Re-map capitalized name → PubChem CID
Mouse_df_file.loc[mask, 'Tail'] = (
    Mouse_df_file.loc[mask, 'Anti-aging interventions'].map(Pubchem_Syn_fil_dict)
)

# Step 4: Again remove trailing '.0' from CID (re-applied after remapping)
Mouse_df_file["Tail"] = Mouse_df_file["Tail"].astype(str).str.replace(r'\.0$', '', regex=True)

# Step 5: Re-attempt IUPAC name lookup for previously unresolved rows
Mouse_df_file.loc[mask, 'Tail_detail_name'] = (
    Mouse_df_file.loc[mask, 'Tail'].map(Pubchem_IUPAC_CID_Dict)
)
Mouse_df_file

# Step 6: Drop rows that still have no resolved chemical (compound not in PubChem)
Mouse_df_file = Mouse_df_file[~Mouse_df_file['Tail_detail_name'].isna()]
Mouse_df_file


,Species,Data type,Head,Anti-aging interventions,Tail,Effect,Publication,2015,Their_Source_PMID,Link,Tail_detail_name
28,Mus musculus,tissue data,GO:1111113,resveratrol,445154,inhibit HDAC activity and prolong lifespan,Histone deacetylase inhibition activity and mo...,2009,19207472.0,https://pubmed.ncbi.nlm.nih.gov/19207472/,"5-[(E)-2-(4-hydroxyphenyl)ethenyl]benzene-1,3-..."
34,Mus musculus,tissue data,GO:1111113,metformin,4091,lifespan extension,AMPK promotes mitochondrial biogenesis and fun...,2017,28143904.0,https://pubmed.ncbi.nlm.nih.gov/28143904/,"3-(diaminomethylidene)-1,1-dimethylguanidine"
37,Mus musculus,tissue data,GO:1111114,taurine,1123,Anti-aging,Potential Anti-aging Role of Taurine via Prope...,2015,25833520.0,https://pubmed.ncbi.nlm.nih.gov/25833520/,2-aminoethanesulfonic acid
38,Mus musculus,tissue data,GO:1111114,4-phenylbutyrate,22053264,improve health span,Reducing ER stress with chaperone therapy reve...,2022,35488730.0,https://pubmed.ncbi.nlm.nih.gov/35488730/,4-phenylbutanoate
53,Mus musculus,tissue data,GO:1111114,nordihydroguaiaretic acid (ndga),4534,enhanced lifespan,Lifespan-increasing drug nordihydroguaiaretic ...,2019,31602311.0,https://pubmed.ncbi.nlm.nih.gov/31602311/,"4-[4-(3,4-dihydroxyphenyl)-2,3-dimethylbutyl]b..."
61,Mus musculus,tissue data,GO:1111115,Growth hormone,170907453,GH may have a beneficial effect on cognition i...,Growth hormone treatment attenuates age-relate...,2004,15489035.0,https://pubmed.ncbi.nlm.nih.gov/15489035/,"(2S)-2-[[(1R,4S,7S,13S,16R,21R,24R,27S,30S,33R..."
66,Mus musculus,tissue data,GO:1111115,Melatonin,896,melatonin induces cognitive enhancement and br...,Melatonin induces mechanisms of brain resilien...,2018,29907977.0,https://pubmed.ncbi.nlm.nih.gov/29907977/,N-[2-(5-methoxy-1H-indol-3-yl)ethyl]acetamide
67,Mus musculus,tissue data,GO:1111115,resveratrol,445154,lifespan extension,The molecular targets of resveratrol,2015,25315298.0,https://pubmed.ncbi.nlm.nih.gov/25315298/,"5-[(E)-2-(4-hydroxyphenyl)ethenyl]benzene-1,3-..."
70,Mus musculus,tissue data,GO:0090398,Dasatinib,3062316,lifespan-extending,The Achilles' heel of senescent cells: from tr...,2015,25754370.0,https://pubmed.ncbi.nlm.nih.gov/25754370/,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...
71,Mus musculus,tissue data,GO:0090398,rapamycin,5284616,"During cell cycle arrest, rapamycin transforme...",Rapamycin decelerates cellular senescence,2009,19471117.0,https://pubmed.ncbi.nlm.nih.gov/19471117/,"(1R,9S,12S,15R,16E,18R,19R,21R,23S,24E,26E,28E..."


In [158]:
# # ─────────────────────────────────────────────────────────────────
# # SWAP Head/Tail so final direction is: Chemical → BiologicalProcess
# # Before swap: Head = GO ID (hallmark), Tail = PubChem CID (chemical)
# # After swap:  Head = PubChem CID (chemical), Tail = GO ID (hallmark)
# # ─────────────────────────────────────────────────────────────────
Mouse_df_file[['Head', 'Tail']] = Mouse_df_file[['Tail', 'Head']]

# # Rename Tail_detail_name → Head_Detail_name after swap (it now describes the Head/chemical)
Mouse_df_file = Mouse_df_file.rename(columns={'Tail_detail_name': 'Head_Detail_name'})
Mouse_df_file

/tmp/ipykernel_2405180/1074181022.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Mouse_df_file[['Head', 'Tail']] = Mouse_df_file[['Tail', 'Head']]


,Species,Data type,Head,Anti-aging interventions,Tail,Effect,Publication,2015,Their_Source_PMID,Link,Head_Detail_name
28,Mus musculus,tissue data,445154,resveratrol,GO:1111113,inhibit HDAC activity and prolong lifespan,Histone deacetylase inhibition activity and mo...,2009,19207472.0,https://pubmed.ncbi.nlm.nih.gov/19207472/,"5-[(E)-2-(4-hydroxyphenyl)ethenyl]benzene-1,3-..."
34,Mus musculus,tissue data,4091,metformin,GO:1111113,lifespan extension,AMPK promotes mitochondrial biogenesis and fun...,2017,28143904.0,https://pubmed.ncbi.nlm.nih.gov/28143904/,"3-(diaminomethylidene)-1,1-dimethylguanidine"
37,Mus musculus,tissue data,1123,taurine,GO:1111114,Anti-aging,Potential Anti-aging Role of Taurine via Prope...,2015,25833520.0,https://pubmed.ncbi.nlm.nih.gov/25833520/,2-aminoethanesulfonic acid
38,Mus musculus,tissue data,22053264,4-phenylbutyrate,GO:1111114,improve health span,Reducing ER stress with chaperone therapy reve...,2022,35488730.0,https://pubmed.ncbi.nlm.nih.gov/35488730/,4-phenylbutanoate
53,Mus musculus,tissue data,4534,nordihydroguaiaretic acid (ndga),GO:1111114,enhanced lifespan,Lifespan-increasing drug nordihydroguaiaretic ...,2019,31602311.0,https://pubmed.ncbi.nlm.nih.gov/31602311/,"4-[4-(3,4-dihydroxyphenyl)-2,3-dimethylbutyl]b..."
61,Mus musculus,tissue data,170907453,Growth hormone,GO:1111115,GH may have a beneficial effect on cognition i...,Growth hormone treatment attenuates age-relate...,2004,15489035.0,https://pubmed.ncbi.nlm.nih.gov/15489035/,"(2S)-2-[[(1R,4S,7S,13S,16R,21R,24R,27S,30S,33R..."
66,Mus musculus,tissue data,896,Melatonin,GO:1111115,melatonin induces cognitive enhancement and br...,Melatonin induces mechanisms of brain resilien...,2018,29907977.0,https://pubmed.ncbi.nlm.nih.gov/29907977/,N-[2-(5-methoxy-1H-indol-3-yl)ethyl]acetamide
67,Mus musculus,tissue data,445154,resveratrol,GO:1111115,lifespan extension,The molecular targets of resveratrol,2015,25315298.0,https://pubmed.ncbi.nlm.nih.gov/25315298/,"5-[(E)-2-(4-hydroxyphenyl)ethenyl]benzene-1,3-..."
70,Mus musculus,tissue data,3062316,Dasatinib,GO:0090398,lifespan-extending,The Achilles' heel of senescent cells: from tr...,2015,25754370.0,https://pubmed.ncbi.nlm.nih.gov/25754370/,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...
71,Mus musculus,tissue data,5284616,rapamycin,GO:0090398,"During cell cycle arrest, rapamycin transforme...",Rapamycin decelerates cellular senescence,2009,19471117.0,https://pubmed.ncbi.nlm.nih.gov/19471117/,"(1R,9S,12S,15R,16E,18R,19R,21R,23S,24E,26E,28E..."


In [159]:
# ─────────────────────────────────────────────────────────────────
# REVERSE LOOKUP: GO ID → Hallmark name (for Tail detail after swap)
# Inverse of aging_hallmarks_to_go; same dict reused from File 4
# ─────────────────────────────────────────────────────────────────
hallmark_to_go = {
    "GO:1111111": "Genomic Instability",
    "GO:1111112": "Telomere Attrition",
    "GO:1111113": "Epigenetic Alterations",
    "GO:1111114": "Loss of Proteostasis",
    "GO:1111115": "Deregulated Nutrient Sensing",
    "GO:1111116": "Mitochondrial Dysfunction",
    "GO:0090398": "Cellular Senescence",        # Official QuickGO term
    "GO:1111117": "Stem Cell Exhaustion",
    "GO:1111118": "Altered Intercellular Communication"
}
Mouse_df_file
# Map GO ID (now in Tail) → human-readable hallmark name
# NOTE: 'tail' (lowercase) will fail here — columns are still mixed case at this point.
# This should be Mouse_df_file['Tail'].map(...) until .str.lower() is applied below.
Mouse_df_file['Tail_detail_name'] = Mouse_df_file['Tail'].map(hallmark_to_go)
Mouse_df_file

,Species,Data type,Head,Anti-aging interventions,Tail,Effect,Publication,2015,Their_Source_PMID,Link,Head_Detail_name,Tail_detail_name
28,Mus musculus,tissue data,445154,resveratrol,GO:1111113,inhibit HDAC activity and prolong lifespan,Histone deacetylase inhibition activity and mo...,2009,19207472.0,https://pubmed.ncbi.nlm.nih.gov/19207472/,"5-[(E)-2-(4-hydroxyphenyl)ethenyl]benzene-1,3-...",Epigenetic Alterations
34,Mus musculus,tissue data,4091,metformin,GO:1111113,lifespan extension,AMPK promotes mitochondrial biogenesis and fun...,2017,28143904.0,https://pubmed.ncbi.nlm.nih.gov/28143904/,"3-(diaminomethylidene)-1,1-dimethylguanidine",Epigenetic Alterations
37,Mus musculus,tissue data,1123,taurine,GO:1111114,Anti-aging,Potential Anti-aging Role of Taurine via Prope...,2015,25833520.0,https://pubmed.ncbi.nlm.nih.gov/25833520/,2-aminoethanesulfonic acid,Loss of Proteostasis
38,Mus musculus,tissue data,22053264,4-phenylbutyrate,GO:1111114,improve health span,Reducing ER stress with chaperone therapy reve...,2022,35488730.0,https://pubmed.ncbi.nlm.nih.gov/35488730/,4-phenylbutanoate,Loss of Proteostasis
53,Mus musculus,tissue data,4534,nordihydroguaiaretic acid (ndga),GO:1111114,enhanced lifespan,Lifespan-increasing drug nordihydroguaiaretic ...,2019,31602311.0,https://pubmed.ncbi.nlm.nih.gov/31602311/,"4-[4-(3,4-dihydroxyphenyl)-2,3-dimethylbutyl]b...",Loss of Proteostasis
61,Mus musculus,tissue data,170907453,Growth hormone,GO:1111115,GH may have a beneficial effect on cognition i...,Growth hormone treatment attenuates age-relate...,2004,15489035.0,https://pubmed.ncbi.nlm.nih.gov/15489035/,"(2S)-2-[[(1R,4S,7S,13S,16R,21R,24R,27S,30S,33R...",Deregulated Nutrient Sensing
66,Mus musculus,tissue data,896,Melatonin,GO:1111115,melatonin induces cognitive enhancement and br...,Melatonin induces mechanisms of brain resilien...,2018,29907977.0,https://pubmed.ncbi.nlm.nih.gov/29907977/,N-[2-(5-methoxy-1H-indol-3-yl)ethyl]acetamide,Deregulated Nutrient Sensing
67,Mus musculus,tissue data,445154,resveratrol,GO:1111115,lifespan extension,The molecular targets of resveratrol,2015,25315298.0,https://pubmed.ncbi.nlm.nih.gov/25315298/,"5-[(E)-2-(4-hydroxyphenyl)ethenyl]benzene-1,3-...",Deregulated Nutrient Sensing
70,Mus musculus,tissue data,3062316,Dasatinib,GO:0090398,lifespan-extending,The Achilles' heel of senescent cells: from tr...,2015,25754370.0,https://pubmed.ncbi.nlm.nih.gov/25754370/,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...,Cellular Senescence
71,Mus musculus,tissue data,5284616,rapamycin,GO:0090398,"During cell cycle arrest, rapamycin transforme...",Rapamycin decelerates cellular senescence,2009,19471117.0,https://pubmed.ncbi.nlm.nih.gov/19471117/,"(1R,9S,12S,15R,16E,18R,19R,21R,23S,24E,26E,28E...",Cellular Senescence


In [160]:
# ─────────────────────────────────────────────────────────────────
# ADD METADATA COLUMNS for KG schema
# ─────────────────────────────────────────────────────────────────
Mouse_df_file['head_type']       = 'ChemicalEntity'
Mouse_df_file['tail_type']       = 'BiologicalProcess'
Mouse_df_file['Relation']        = 'ChemicalEntity_BiologicalProcess'
Mouse_df_file['head_id_is']      = 'Pubchem'    # ID namespace for head node
Mouse_df_file['tail_id_is']      = 'Quick_GO'   # ID namespace for tail node
Mouse_df_file['relation_source'] = 'AgeAnnoMO'  # Source database
Mouse_df_file


# ─────────────────────────────────────────────────────────────────
# FINAL CLEANUP AND COLUMN ORDERING
# ─────────────────────────────────────────────────────────────────

# Define desired column order for output (all lowercase — applied below)
desired_order = [
    'head', 'relation', 'tail',
    'head_type', 'tail_type',
    'head_detail_name', 'tail_detail_name',
    'head_id_is', 'tail_id_is',
    'relation_source', 'their_source_pmid', 'species'
]
Mouse_df_file

# NOTE: 'Relation' is set twice to the same value — the second assignment below is redundant
Mouse_df_file['Relation'] = 'ChemicalEntity_BiologicalProcess'


In [161]:
# Standardize all column names to lowercase for consistency across KG files
Mouse_df_file.columns = Mouse_df_file.columns.str.lower()
Mouse_df_file

# Reorder columns (requires all columns to already be lowercase 
Mouse_df_file = Mouse_df_file[desired_order]


# Drop any duplicate columns (can arise from repeated renames/merges)
Mouse_df_file = Mouse_df_file.loc[:, ~Mouse_df_file.columns.duplicated()]

# Drop rows where Tail is NaN (extra safety filter after column reorder)
Mouse_df_file = Mouse_df_file[~Mouse_df_file['tail'].isna()]
Mouse_df_file

# Save processed Mouse Chemical→BiologicalProcess edges to CSV
Mouse_df_file.to_csv(f'{OUT_PATH}ageanno_processed_mo/AgeAnnoMO_Mouse_Chemical_BiologicalProcess_1.csv', index=None)

In [162]:
Mouse_df_file

,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,head_id_is,tail_id_is,relation_source,their_source_pmid,species
28,445154,ChemicalEntity_BiologicalProcess,GO:1111113,ChemicalEntity,BiologicalProcess,"5-[(E)-2-(4-hydroxyphenyl)ethenyl]benzene-1,3-...",Epigenetic Alterations,Pubchem,Quick_GO,AgeAnnoMO,19207472.0,Mus musculus
34,4091,ChemicalEntity_BiologicalProcess,GO:1111113,ChemicalEntity,BiologicalProcess,"3-(diaminomethylidene)-1,1-dimethylguanidine",Epigenetic Alterations,Pubchem,Quick_GO,AgeAnnoMO,28143904.0,Mus musculus
37,1123,ChemicalEntity_BiologicalProcess,GO:1111114,ChemicalEntity,BiologicalProcess,2-aminoethanesulfonic acid,Loss of Proteostasis,Pubchem,Quick_GO,AgeAnnoMO,25833520.0,Mus musculus
38,22053264,ChemicalEntity_BiologicalProcess,GO:1111114,ChemicalEntity,BiologicalProcess,4-phenylbutanoate,Loss of Proteostasis,Pubchem,Quick_GO,AgeAnnoMO,35488730.0,Mus musculus
53,4534,ChemicalEntity_BiologicalProcess,GO:1111114,ChemicalEntity,BiologicalProcess,"4-[4-(3,4-dihydroxyphenyl)-2,3-dimethylbutyl]b...",Loss of Proteostasis,Pubchem,Quick_GO,AgeAnnoMO,31602311.0,Mus musculus
61,170907453,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,"(2S)-2-[[(1R,4S,7S,13S,16R,21R,24R,27S,30S,33R...",Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,15489035.0,Mus musculus
66,896,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,N-[2-(5-methoxy-1H-indol-3-yl)ethyl]acetamide,Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,29907977.0,Mus musculus
67,445154,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,"5-[(E)-2-(4-hydroxyphenyl)ethenyl]benzene-1,3-...",Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,25315298.0,Mus musculus
70,3062316,ChemicalEntity_BiologicalProcess,GO:0090398,ChemicalEntity,BiologicalProcess,N-(2-chloro-6-methylphenyl)-2-[[6-[4-(2-hydrox...,Cellular Senescence,Pubchem,Quick_GO,AgeAnnoMO,25754370.0,Mus musculus
71,5284616,ChemicalEntity_BiologicalProcess,GO:0090398,ChemicalEntity,BiologicalProcess,"(1R,9S,12S,15R,16E,18R,19R,21R,23S,24E,26E,28E...",Cellular Senescence,Pubchem,Quick_GO,AgeAnnoMO,19471117.0,Mus musculus


# 6

In [139]:
# ─────────────────────────────────────────────────────────────────
# LOAD: Dysregulated Metabolites in Aging (AgeAnnoMO)
# Source: Differential_metabolite.xlsx — contains metabolites with
# flags indicating whether they are age-related
# ─────────────────────────────────────────────────────────────────
Ag_Dysregulated = pd.read_excel(f'{BASE_PATH}ageannomo/AgeAnnoMO-main/Dysregulated_metabolism_in_aging/Differential_metabolite.xlsx'
)
Ag_Dysregulated

# Keep only rows confirmed as age-related metabolites
Ag_Dysregulated = Ag_Dysregulated[Ag_Dysregulated['is_AgeRelatedMetabolite'] == True]

# Inspect which organisms are present in the filtered data
print(set(Ag_Dysregulated['Animal']))


# ─────────────────────────────────────────────────────────────────
# RENAME COLUMNS to match KG schema
# ─────────────────────────────────────────────────────────────────
Ag_Dysregulated = Ag_Dysregulated.rename(columns={
    'Name':                  'Head',    # Metabolite name → Head node
    'is_AgeRelatedMetabolite': 'Tail',  # Will be replaced by GO ID below
    'Animal':                'Species'
})


# ─────────────────────────────────────────────────────────────────
# ASSIGN Tail → GO:1111115 (Deregulated Nutrient Sensing)
# All dysregulated metabolites are linked to this aging hallmark.
# GO:1111115 is a custom internal ID (not in QuickGO); see aging_hallmarks_to_go.
# NOTE: Tail_detail_name is set twice below — first assignment is overwritten,
# only the empty string version at the end persists (filled per-species later).
# ─────────────────────────────────────────────────────────────────
Ag_Dysregulated['Tail']             = 'GO:1111115'
Ag_Dysregulated['Tail_detail_name'] = 'GO:1111115'   # Overwritten below — placeholder
Ag_Dysregulated['Tail_id']          = 'Dysregulated_metabolism'


# ─────────────────────────────────────────────────────────────────
# ADD METADATA COLUMNS for KG schema
# ─────────────────────────────────────────────────────────────────
Ag_Dysregulated['Head_type']      = 'ChemicalEntity'
Ag_Dysregulated['Tail_type']      = 'BiologicalProcess'
Ag_Dysregulated['Relation']       = 'ChemicalEntity_BiologicalProcess'  
                                                        
                                                        
Ag_Dysregulated['Relation_Source'] = 'AgeAnnoMO'


# ─────────────────────────────────────────────────────────────────
# STANDARDIZE Species names to full scientific names
# ─────────────────────────────────────────────────────────────────
Ag_Dysregulated.loc[Ag_Dysregulated['Species'] == 'Human', 'Species'] = 'Homo sapiens'
Ag_Dysregulated.loc[Ag_Dysregulated['Species'] == 'Mouse', 'Species'] = 'Mus musculus'


# ─────────────────────────────────────────────────────────────────
# INITIALIZE detail/ID columns as empty strings
# These will be filled in per-species after PubChem lookup below
# ─────────────────────────────────────────────────────────────────
Ag_Dysregulated['Head_detail_name'] = ''

Ag_Dysregulated['Head_ID_IS']       = ''
Ag_Dysregulated['Tail_ID_IS']       = ''


# ─────────────────────────────────────────────────────────────────
# REORDER columns to match KG output schema and drop duplicates
# ─────────────────────────────────────────────────────────────────
desired_order = [
    'Head', 'Relation', 'Tail',
    'Head_type', 'Tail_type',
    'Head_detail_name', 'Tail_detail_name',
    'Head_ID_IS', 'Tail_ID_IS',
    'Relation_Source', 'Species'
]
Ag_Dysregulated = Ag_Dysregulated[desired_order]
Ag_Dysregulated = Ag_Dysregulated.drop_duplicates()
Ag_Dysregulated


# ─────────────────────────────────────────────────────────────────
# HUMAN SUBSET: Resolve metabolite names → PubChem CIDs → IUPAC names
# ─────────────────────────────────────────────────────────────────

# Filter to human rows only
Human = Ag_Dysregulated[Ag_Dysregulated['Species'] == 'Homo sapiens']

# Map metabolite name (Head) → PubChem CID using synonym dictionary
Human['Head_id'] = Human['Head'].map(Pubchem_Syn_fil_dict)

# Remove trailing '.0' from CID values (artifact of float conversion)
Human["Head_id"] = Human["Head_id"].astype(str).str.replace(r'\.0$', '', regex=True)
Human

# Map PubChem CID → IUPAC name for human-readable chemical identity
Human['Head_detail_name'] = Human['Head_id'].map(Pubchem_IUPAC_CID_Dict)

# Map Tail GO ID → hallmark name using reverse lookup dict (hallmark_to_go)
Human['Tail_detail_name'] = Human['Tail'].map(hallmark_to_go)

# Set ID namespace labels
Human['Head_ID_IS'] = 'Pubchem'
Human['Tail_ID_IS'] = 'Quick_GO'
Human


# ─────────────────────────────────────────────────────────────────
# SWAP Head/Head_id so Head holds the PubChem CID (numeric ID)
# and Head_id holds the original metabolite name (now redundant)
# Before swap: Head = metabolite name, Head_id = PubChem CID
# After swap:  Head = PubChem CID,     Head_id = metabolite name
# ─────────────────────────────────────────────────────────────────
Human[['Head_id', 'Head']] = Human[['Head', 'Head_id']]

# Drop rows where PubChem CID was not resolved (Head = 'nan' string after swap)
Human = Human[Human['Head'] != 'nan']
Human

# Confirm relation type label for this file
Human['Relation'] = 'ChemicalEntity_BiologicalProcess'
Human

# Save processed Human Dysregulated Metabolite → BiologicalProcess edges to CSV
Human.to_csv(f'{OUT_PATH}ageanno_processed_mo/AgeAnnoMO_Human_Chemical_BiologicalProcess_2.csv', index=None)
Human

{'Human', 'Mouse'}


/tmp/ipykernel_2405180/1920598507.py:91: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Human['Head_id'] = Human['Head'].map(Pubchem_Syn_fil_dict)
/tmp/ipykernel_2405180/1920598507.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Human["Head_id"] = Human["Head_id"].astype(str).str.replace(r'\.0$', '', regex=True)
/tmp/ipykernel_2405180/1920598507.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See 

,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Relation_Source,Species,Head_id
2818,80220,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,"1-methyl-3,7-dihydropurine-2,6-dione",Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Homo sapiens,1-Methylxanthine
2825,700653,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,(2S)-2-acetamido-3-(1H-indol-3-yl)propanoic acid,Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Homo sapiens,N-Acetyl-L-tryptophan
2829,464,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,2-benzamidoacetic acid,Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Homo sapiens,Hippuric acid
2845,6262,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,"(2S)-2,5-diaminopentanoic acid",Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Homo sapiens,L-ornithine
2848,7405,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,(2S)-5-oxopyrrolidine-2-carboxylic acid,Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Homo sapiens,L-Pyroglutamic acid
2857,7018721,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,(2S)-2-amino-5-[[(2S)-1-(carboxymethylamino)-1...,Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Homo sapiens,Ophthalmic acid
2859,588,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,2-amino-3-methyl-4H-imidazol-5-one,Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Homo sapiens,2-Imino-1-methylimidazolidin-4-one
2860,611,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,2-aminopentanedioic acid,Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Homo sapiens,DL-Glutamic acid
2872,168379,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,(3R)-3-(2-methylpropanoyloxy)-4-(trimethylazan...,Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Homo sapiens,Isobutyryl-L-carnitine
2876,6426851,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,3-(3-methylbutanoyloxy)-4-(trimethylazaniumyl)...,Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Homo sapiens,Isovalerylcarnitine


In [140]:
# ─────────────────────────────────────────────────────────────────
# MOUSE SUBSET: Resolve metabolite names → PubChem CIDs → IUPAC names
# Same pipeline as Human subset above, but for Mus musculus
# ─────────────────────────────────────────────────────────────────

# Filter to mouse rows only
# NOTE: Add .copy() to avoid SettingWithCopyWarning on slice assignments below
Mouse = Ag_Dysregulated[Ag_Dysregulated['Species'] == 'Mus musculus'].copy()

# Map metabolite name (Head) → PubChem CID using synonym dictionary
Mouse['Head_id'] = Mouse['Head'].map(Pubchem_Syn_fil_dict)

# Remove trailing '.0' from CID values (artifact of float conversion)
Mouse["Head_id"] = Mouse["Head_id"].astype(str).str.replace(r'\.0$', '', regex=True)

# Map PubChem CID → IUPAC name for human-readable chemical identity
Mouse['Head_detail_name'] = Mouse['Head_id'].map(Pubchem_IUPAC_CID_Dict)

# Map Tail GO ID → hallmark name using reverse lookup dict (hallmark_to_go)
Mouse['Tail_detail_name'] = Mouse['Tail'].map(hallmark_to_go)

# Set ID namespace labels
Mouse['Head_ID_IS'] = 'Pubchem'
Mouse['Tail_ID_IS'] = 'Quick_GO'


# ─────────────────────────────────────────────────────────────────
# SWAP Head/Head_id so Head holds the PubChem CID (numeric ID)
# and Head_id holds the original metabolite name (now redundant)
# Before swap: Head = metabolite name, Head_id = PubChem CID
# After swap:  Head = PubChem CID,     Head_id = metabolite name
# ─────────────────────────────────────────────────────────────────
Mouse[['Head_id', 'Head']] = Mouse[['Head', 'Head_id']]

# Drop rows where PubChem CID was not resolved (Head = 'nan' string after swap)
Mouse = Mouse[Mouse['Head'] != 'nan']
Mouse

# Confirm relation type label for this file
Mouse['Relation'] = 'ChemicalEntity_BiologicalProcess'
Mouse

# Save processed Mouse Dysregulated Metabolite → BiologicalProcess edges to CSV
Mouse.to_csv(f'{OUT_PATH}ageanno_processed_mo/AgeAnnoMO_Mouse_Chemical_BiologicalProcess_2.csv', index=None)

In [141]:
Mouse

,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Relation_Source,Species,Head_id
11,10917,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,(3R)-3-hydroxy-4-(trimethylazaniumyl)butanoate,Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Mus musculus,Levocarnitine
1593,657272,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,"[(2R)-2,3-dihydroxypropyl] 2-(trimethylazanium...",Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Mus musculus,Choline Alfoscerate
2611,288,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,3-hydroxy-4-(trimethylazaniumyl)butanoate,Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Mus musculus,DL-Carnitine
2705,107738,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,3-propanoyloxy-4-(trimethylazaniumyl)butanoate,Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Mus musculus,Propionylcarnitine
2772,5283469,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,"2,3-dihydroxypropyl (9Z,12Z)-octadeca-9,12-die...",Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Mus musculus,Glyceryl monolinoleate
2777,5460358,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,"(2R)-2,3-dihydroxypropanoate",Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Mus musculus,D-glycerate
2779,40489139,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,"[(2R)-2,3-dihydroxypropyl] (9Z,12Z)-octadeca-9...",Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Mus musculus,1-Monolinolein
2782,6436630,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,"[(2S)-2,3-dihydroxypropyl] (9Z,12Z)-octadeca-9...",Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Mus musculus,alpha-Glyceryl linoleate
2786,7045767,ChemicalEntity_BiologicalProcess,GO:1111115,ChemicalEntity,BiologicalProcess,(3R)-3-acetyloxy-4-(trimethylazaniumyl)butanoate,Deregulated Nutrient Sensing,Pubchem,Quick_GO,AgeAnnoMO,Mus musculus,Acetyl-L-carnitine


# 7

In [4]:
Ag_Metabolite_interaction = pd.read_excel(f'{BASE_PATH}ageannomo/AgeAnnoMO-main/Dysregulated_metabolism_in_aging/Metabolite_interaction.xlsx')
Ag_Metabolite_interaction

,Dataset ID,Animal,Tissue,metabolite1,metabolite2,r,p,pFDR
0,AMO-ME-001,Mouse,Brain (cerebral cortex),Adenosine monophosphate,(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,-0.325893,0.009148,0.012057
1,AMO-ME-001,Mouse,Brain (cerebral cortex),(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,Cholesteryl linoleate,-0.636137,0.0,0.0
2,AMO-ME-001,Mouse,Brain (cerebral cortex),Adenosine monophosphate,N-(9Z-octadecenoyl)-sphinganine,-0.270929,0.031736,0.039924
3,AMO-ME-001,Mouse,Brain (cerebral cortex),N-(9Z-octadecenoyl)-sphinganine,N-icosanoylsphingosine,0.880088,0,0
4,AMO-ME-001,Mouse,Brain (cerebral cortex),N-(9Z-octadecenoyl)-sphinganine,"N-[(E)-1,3-dihydroxyicos-4-en-2-yl]octadecanamide",0.800307,0.0,0.0
...,...,...,...,...,...,...,...,...
275130,AMO-ME-015,Human,Plasma,"(2S,4R)-4-hydroxy-1-[(2S)-pyrrolidin-1-ium-2-c...",Chenodeoxycholic acid 3-sulfate,-0.178555,0.003357,0.011189
275131,AMO-ME-015,Human,Plasma,L-Homoarginine,Chenodeoxycholic acid 3-sulfate,0.181466,0.002867,0.009796
275132,AMO-ME-015,Human,Plasma,5-Acetylamino-6-amino-3-methyluracil,Chenodeoxycholic acid 3-sulfate,0.180007,0.003104,0.010458
275133,AMO-ME-015,Human,Plasma,11-beta-Hydroxyandrosterone-3-glucuronide,Chenodeoxycholic acid 3-sulfate,0.285454,0.000002,0.000016


In [6]:
min(Ag_Metabolite_interaction.astype(str)['p']), max(Ag_Metabolite_interaction['p'].astype(str))

('0', 'p')

In [142]:
# ─────────────────────────────────────────────────────────────────
# LOAD: Metabolite-Metabolite Interactions in Aging (AgeAnnoMO)
# Source: Metabolite_interaction.xlsx — pairwise interactions between
# dysregulated metabolites observed during aging
# ─────────────────────────────────────────────────────────────────
Ag_Metabolite_interaction = pd.read_excel(f'{BASE_PATH}ageannomo/AgeAnnoMO-main/Dysregulated_metabolism_in_aging/Metabolite_interaction.xlsx')

# Inspect which organisms are present in the data
print(set(Ag_Metabolite_interaction['Animal']))

# ─────────────────────────────────────────────────────────────────
# RENAME COLUMNS to match KG schema
# Both Head and Tail are metabolites (ChemicalEntity → ChemicalEntity)
# ─────────────────────────────────────────────────────────────────
Ag_Metabolite_interaction = Ag_Metabolite_interaction.rename(columns={
    'metabolite1': 'Head',
    'metabolite2': 'Tail',
    'Animal':      'Species'
})


# ─────────────────────────────────────────────────────────────────
# ADD METADATA COLUMNS for KG schema
# Both nodes are chemical entities identified by PubChem CIDs
# ─────────────────────────────────────────────────────────────────
Ag_Metabolite_interaction['Head_type']       = 'ChemicalEntity'
Ag_Metabolite_interaction['Tail_type']       = 'ChemicalEntity'
Ag_Metabolite_interaction['Relation']        = 'ChemicalEntity_ChemicalEntity'
Ag_Metabolite_interaction['Head_id_is']      = 'Pubchem'
Ag_Metabolite_interaction['Tail_id_is']      = 'Pubchem'
Ag_Metabolite_interaction['Head_detail_name'] = ''   # Filled per-species below
Ag_Metabolite_interaction['Tail_detail_name'] = ''   # Filled per-species below
Ag_Metabolite_interaction['Relation_Source'] = 'AgeAnnoMO'


# ─────────────────────────────────────────────────────────────────
# REORDER columns to match KG output schema and drop duplicates
# ─────────────────────────────────────────────────────────────────
desired_order = [
    'Head', 'Relation', 'Tail',
    'Head_type', 'Tail_type',
    'Head_detail_name', 'Tail_detail_name',
    'Head_id_is', 'Tail_id_is',
    'Relation_Source', 'Species'
]
Ag_Metabolite_interaction = Ag_Metabolite_interaction[desired_order]
Ag_Metabolite_interaction = Ag_Metabolite_interaction.drop_duplicates()
Ag_Metabolite_interaction


# ═════════════════════════════════════════════════════════════════
# HUMAN SUBSET: Resolve both metabolite names → PubChem CIDs
# ═════════════════════════════════════════════════════════════════

# Filter to human rows only; .copy() prevents SettingWithCopyWarning
Human = Ag_Metabolite_interaction[Ag_Metabolite_interaction['Species'] == 'Human'].copy()

# ── HEAD (metabolite1) ──────────────────────────────────────────
# Map metabolite name → PubChem CID using synonym dictionary
Human['Head_id'] = Human['Head'].map(Pubchem_Syn_fil_dict)

# Remove trailing '.0' from CID (artifact of float conversion)
Human["Head_id"] = Human["Head_id"].astype(str).str.replace(r'\.0$', '', regex=True)

# Map PubChem CID → IUPAC name for human-readable chemical identity
Human['Head_detail_name'] = Human['Head_id'].map(Pubchem_IUPAC_CID_Dict)

# Set Head ID namespace
Human['Head_ID_IS'] = 'Pubchem'

# Swap: Head ← PubChem CID, Head_id ← original metabolite name
# After swap: Head = PubChem CID (numeric ID used in KG)
Human[['Head_id', 'Head']] = Human[['Head', 'Head_id']]

# Drop rows where Head CID was not resolved
Human = Human[Human['Head'] != 'nan']

# ── TAIL (metabolite2) ──────────────────────────────────────────
# Map metabolite name → PubChem CID using synonym dictionary
Human['Tail_id'] = Human['Tail'].map(Pubchem_Syn_fil_dict)

# Remove trailing '.0' from CID (artifact of float conversion)
Human["Tail_id"] = Human["Tail_id"].astype(str).str.replace(r'\.0$', '', regex=True)

# Map PubChem CID → IUPAC name for human-readable chemical identity
Human['Tail_detail_name'] = Human['Tail_id'].map(Pubchem_IUPAC_CID_Dict)

# Set Tail ID namespace
Human['Tail_ID_IS'] = 'Pubchem'

# Swap: Tail ← PubChem CID, Tail_id ← original metabolite name
# After swap: Tail = PubChem CID (numeric ID used in KG)
Human[['Tail_id', 'Tail']] = Human[['Tail', 'Tail_id']]

# Drop rows where Tail CID was not resolved
Human = Human[Human['Tail'] != 'nan']
Human

# Save processed Human Metabolite↔Metabolite edges to CSV
Human.to_csv(f'{OUT_PATH}ageanno_processed_mo/AgeAnnoMO_Human_Chemical_Chemical_1.csv', index=None)
Human

{'Human', 'Mouse'}


,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Tail_detail_name,Head_id_is,Tail_id_is,Relation_Source,Species,Head_id,Head_ID_IS,Tail_id,Tail_ID_IS
273562,69726,ChemicalEntity_ChemicalEntity,80220,ChemicalEntity,ChemicalEntity,"1-methyl-7,9-dihydro-3H-purine-2,6,8-trione","1-methyl-3,7-dihydropurine-2,6-dione",Pubchem,Pubchem,AgeAnnoMO,Human,1-Methyluric acid,Pubchem,1-Methylxanthine,Pubchem
273573,69726,ChemicalEntity_ChemicalEntity,700653,ChemicalEntity,ChemicalEntity,"1-methyl-7,9-dihydro-3H-purine-2,6,8-trione",(2S)-2-acetamido-3-(1H-indol-3-yl)propanoic acid,Pubchem,Pubchem,AgeAnnoMO,Human,1-Methyluric acid,Pubchem,N-Acetyl-L-tryptophan,Pubchem
273574,80220,ChemicalEntity_ChemicalEntity,700653,ChemicalEntity,ChemicalEntity,"1-methyl-3,7-dihydropurine-2,6-dione",(2S)-2-acetamido-3-(1H-indol-3-yl)propanoic acid,Pubchem,Pubchem,AgeAnnoMO,Human,1-Methylxanthine,Pubchem,N-Acetyl-L-tryptophan,Pubchem
273577,80220,ChemicalEntity_ChemicalEntity,107461,ChemicalEntity,ChemicalEntity,"1-methyl-3,7-dihydropurine-2,6-dione","N-[1-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydroxyme...",Pubchem,Pubchem,AgeAnnoMO,Human,1-Methylxanthine,Pubchem,N4-Acetylcytidine,Pubchem
273594,107461,ChemicalEntity_ChemicalEntity,108192,ChemicalEntity,ChemicalEntity,"N-[1-[(2R,3R,4S,5R)-3,4-dihydroxy-5-(hydroxyme...","(2S,3S,4S,5R,6R)-6-[[(8R,9S,10R,13S,14S,17S)-1...",Pubchem,Pubchem,AgeAnnoMO,Human,N4-Acetylcytidine,Pubchem,Testosterone glucuronide,Pubchem
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275129,154035,ChemicalEntity_ChemicalEntity,21252312,ChemicalEntity,ChemicalEntity,"(2S,3S,4S,5R,6S)-3,4,5-trihydroxy-6-(4-methylp...","(4R)-4-[(3R,5R,7R,8R,9S,10S,13R,14S,17R)-7-hyd...",Pubchem,Pubchem,AgeAnnoMO,Human,p-Cresol glucuronide,Pubchem,Chenodeoxycholic acid 3-sulfate,Pubchem
275130,11902892,ChemicalEntity_ChemicalEntity,21252312,ChemicalEntity,ChemicalEntity,"(2S,4R)-4-hydroxy-1-[(2S)-pyrrolidin-1-ium-2-c...","(4R)-4-[(3R,5R,7R,8R,9S,10S,13R,14S,17R)-7-hyd...",Pubchem,Pubchem,AgeAnnoMO,Human,"(2S,4R)-4-hydroxy-1-[(2S)-pyrrolidin-1-ium-2-c...",Pubchem,Chenodeoxycholic acid 3-sulfate,Pubchem
275131,9085,ChemicalEntity_ChemicalEntity,21252312,ChemicalEntity,ChemicalEntity,(2S)-2-amino-6-(diaminomethylideneamino)hexano...,"(4R)-4-[(3R,5R,7R,8R,9S,10S,13R,14S,17R)-7-hyd...",Pubchem,Pubchem,AgeAnnoMO,Human,L-Homoarginine,Pubchem,Chenodeoxycholic acid 3-sulfate,Pubchem
275132,88299,ChemicalEntity_ChemicalEntity,21252312,ChemicalEntity,ChemicalEntity,"N-(6-amino-3-methyl-2,4-dioxo-1H-pyrimidin-5-y...","(4R)-4-[(3R,5R,7R,8R,9S,10S,13R,14S,17R)-7-hyd...",Pubchem,Pubchem,AgeAnnoMO,Human,5-Acetylamino-6-amino-3-methyluracil,Pubchem,Chenodeoxycholic acid 3-sulfate,Pubchem


In [143]:
# ═════════════════════════════════════════════════════════════════
# MOUSE SUBSET: Resolve both metabolite names → PubChem CIDs
# Identical pipeline to Human subset above, for Mus musculus
# ═════════════════════════════════════════════════════════════════

# Filter to mouse rows only; .copy() prevents SettingWithCopyWarning
Mouse = Ag_Metabolite_interaction[Ag_Metabolite_interaction['Species'] == 'Mouse'].copy()

# ── HEAD (metabolite1) ──────────────────────────────────────────
# Map metabolite name → PubChem CID using synonym dictionary
Mouse['Head_id'] = Mouse['Head'].map(Pubchem_Syn_fil_dict)

# Remove trailing '.0' from CID (artifact of float conversion)
Mouse["Head_id"] = Mouse["Head_id"].astype(str).str.replace(r'\.0$', '', regex=True)

# Map PubChem CID → IUPAC name for human-readable chemical identity
Mouse['Head_detail_name'] = Mouse['Head_id'].map(Pubchem_IUPAC_CID_Dict)

# Set Head ID namespace
Mouse['Head_ID_IS'] = 'Pubchem'

# Swap: Head ← PubChem CID, Head_id ← original metabolite name
Mouse[['Head_id', 'Head']] = Mouse[['Head', 'Head_id']]

# Drop rows where Head CID was not resolved
Mouse = Mouse[Mouse['Head'] != 'nan']

# ── TAIL (metabolite2) ──────────────────────────────────────────
# Map metabolite name → PubChem CID using synonym dictionary
Mouse['Tail_id'] = Mouse['Tail'].map(Pubchem_Syn_fil_dict)

# Remove trailing '.0' from CID (artifact of float conversion)
Mouse["Tail_id"] = Mouse["Tail_id"].astype(str).str.replace(r'\.0$', '', regex=True)

# Map PubChem CID → IUPAC name for human-readable chemical identity
Mouse['Tail_detail_name'] = Mouse['Tail_id'].map(Pubchem_IUPAC_CID_Dict)

# Set Tail ID namespace
Mouse['Tail_ID_IS'] = 'Pubchem'

# Swap: Tail ← PubChem CID, Tail_id ← original metabolite name
Mouse[['Tail_id', 'Tail']] = Mouse[['Tail', 'Tail_id']]

# Drop rows where Tail CID was not resolved
Mouse = Mouse[Mouse['Tail'] != 'nan']
Mouse

# Save processed Mouse Metabolite↔Metabolite edges to CSV
Mouse.to_csv(f'{OUT_PATH}ageanno_processed_mo/AgeAnnoMO_Mouse_CHemical_Chemical.csv', index=None)
Mouse

,Head,Relation,Tail,Head_type,Tail_type,Head_detail_name,Tail_detail_name,Head_id_is,Tail_id_is,Relation_Source,Species,Head_id,Head_ID_IS,Tail_id,Tail_ID_IS
0,6083,ChemicalEntity_ChemicalEntity,28782,ChemicalEntity,ChemicalEntity,"[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-3,4-dihyd...",(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,Pubchem,Pubchem,AgeAnnoMO,Mouse,Adenosine monophosphate,Pubchem,(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,Pubchem
1,28782,ChemicalEntity_ChemicalEntity,5287939,ChemicalEntity,ChemicalEntity,(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,"[(3S,8S,9S,10R,13R,14S,17R)-10,13-dimethyl-17-...",Pubchem,Pubchem,AgeAnnoMO,Mouse,(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,Pubchem,Cholesteryl linoleate,Pubchem
2,6083,ChemicalEntity_ChemicalEntity,6442676,ChemicalEntity,ChemicalEntity,"[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-3,4-dihyd...","(Z)-N-[(2S,3R)-1,3-dihydroxyoctadecan-2-yl]oct...",Pubchem,Pubchem,AgeAnnoMO,Mouse,Adenosine monophosphate,Pubchem,N-(9Z-octadecenoyl)-sphinganine,Pubchem
3,6442676,ChemicalEntity_ChemicalEntity,5283566,ChemicalEntity,ChemicalEntity,"(Z)-N-[(2S,3R)-1,3-dihydroxyoctadecan-2-yl]oct...","N-[(E,2S,3R)-1,3-dihydroxyoctadec-4-en-2-yl]ic...",Pubchem,Pubchem,AgeAnnoMO,Mouse,N-(9Z-octadecenoyl)-sphinganine,Pubchem,N-icosanoylsphingosine,Pubchem
12,28782,ChemicalEntity_ChemicalEntity,102615,ChemicalEntity,ChemicalEntity,(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,(3-hydroxy-2-octadecanoyloxypropyl) octadecanoate,Pubchem,Pubchem,AgeAnnoMO,Mouse,(2S)-2-amino-5-(diaminomethylideneazaniumyl)pe...,Pubchem,"1,2-Distearoyl-rac-glycerol",Pubchem
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273411,439918,ChemicalEntity_ChemicalEntity,9934,ChemicalEntity,ChemicalEntity,(2S)-2-(diaminomethylideneamino)butanedioic acid,"1,3-thiazolidine-4-carboxylic acid",Pubchem,Pubchem,AgeAnnoMO,Mouse,MFCD00055819,Pubchem,Timonacic,Pubchem
273557,866,ChemicalEntity_ChemicalEntity,123372,ChemicalEntity,ChemicalEntity,"2,6-diaminohexanoic acid",ethyl 4-isothiocyanatobutanoate,Pubchem,Pubchem,AgeAnnoMO,Mouse,DL-Lysine,Pubchem,Ethyl 4-isothiocyanatobutanoate,Pubchem
273558,849,ChemicalEntity_ChemicalEntity,123372,ChemicalEntity,ChemicalEntity,piperidine-2-carboxylic acid,ethyl 4-isothiocyanatobutanoate,Pubchem,Pubchem,AgeAnnoMO,Mouse,Pipecolic acid,Pubchem,Ethyl 4-isothiocyanatobutanoate,Pubchem
273559,111306,ChemicalEntity_ChemicalEntity,123372,ChemicalEntity,ChemicalEntity,(2S)-pyrrolidine-2-carboxamide,ethyl 4-isothiocyanatobutanoate,Pubchem,Pubchem,AgeAnnoMO,Mouse,Prolinamide,Pubchem,Ethyl 4-isothiocyanatobutanoate,Pubchem


# 8

In [144]:
# ─────────────────────────────────────────────────────────────────
# LOAD: Aging-related Differentially Methylated Regions (DMRs)
# Source: Aging-related DMR.xlsx — genes with epigenetic modifications
# (methylation changes) observed during aging, from AgeAnnoMO
# ─────────────────────────────────────────────────────────────────
Epigenetic_alteration = pd.read_excel(f'{BASE_PATH}ageannomo/AgeAnnoMO-main/Epigenetic alterations/Aging-related DMR.xlsx'
)

# Keep only Human and Mouse; other organisms excluded for this KG build
Sp_list = ['Mouse', 'Human']
Epigenetic_alteration = Epigenetic_alteration[Epigenetic_alteration['Animal'].isin(Sp_list)]

# Inspect species and methylation types present after filtering
print(set(Epigenetic_alteration['Animal']))
print(set(Epigenetic_alteration['Methy.type']))


# ─────────────────────────────────────────────────────────────────
# RENAME COLUMNS to match KG schema
# Head = gene symbol, Tail = methylation type (replaced by GO ID below)
# ─────────────────────────────────────────────────────────────────
Epigenetic_alteration = Epigenetic_alteration.rename(columns={
    'Symbol':     'Head',   # Gene symbol
    'Methy.type': 'Tail',   # Methylation type → will be replaced by GO:1111113
    'Animal':     'Species'
})


# ─────────────────────────────────────────────────────────────────
# ADD METADATA COLUMNS for KG schema
# Relation links a Gene to the Epigenetic Alterations aging hallmark
# NOTE: Tail_type uses 'Epigenitic_modification' (typo) — consider
# standardizing to 'BiologicalProcess' to match other files,
# since Tail is replaced by GO:1111113 (BiologicalProcess) below
# ─────────────────────────────────────────────────────────────────
Epigenetic_alteration['Head_type']       = 'Gene'
Epigenetic_alteration['Tail_type']       = 'Epigenitic_modification'  # NOTE: typo in original
Epigenetic_alteration['Relation']        = 'Gene-Hallmark(epigenetic alterations)'
Epigenetic_alteration['Relation_Source'] = 'AgeAnnoMO'


# ─────────────────────────────────────────────────────────────────
# REORDER columns to match KG output schema and drop duplicates
# ─────────────────────────────────────────────────────────────────
desired_order = [
    'Head', 'Relation', 'Tail',
    'Head_type', 'Tail_type',
    'Relation_Source', 'Species'
]
Epigenetic_alteration = Epigenetic_alteration[desired_order]
Epigenetic_alteration = Epigenetic_alteration.drop_duplicates()
Epigenetic_alteration

# Confirm species present after deduplication
set(Epigenetic_alteration['Species'])


# ═════════════════════════════════════════════════════════════════
# HUMAN SUBSET: Gene → Epigenetic Alterations (BiologicalProcess)
# ═════════════════════════════════════════════════════════════════

# Filter to human rows only; .copy() prevents SettingWithCopyWarning
Human = Epigenetic_alteration[Epigenetic_alteration['Species'] == 'Human'].copy()

# Replace raw methylation type with GO ID for "Epigenetic alteration" hallmark
# GO:1111113 is a custom internal ID (not in QuickGO); see aging_hallmarks_to_go
Human['Tail']             = 'GO:1111113'
Human['Tail_detail_name'] = 'Epigenetic Alterations'

# Update relation label to standard KG edge type
Human['Relation'] = 'Gene_BiologicalProcess'

# Map human gene symbol → NCBI gene description using human symbol dict
Human['Head_detail_name'] = Human['Head'].map(NCBI_ALL_Symb_Desc_dict)
Human

# Drop rows where gene symbol had no NCBI match (unrecognized or non-standard symbols)
Human = Human[~Human['Head_detail_name'].isna()]
Human

# Set ID namespace labels
Human['Head_id_is'] = 'NCBI_ID'   # Gene identified by NCBI Gene ID
Human['Tail_id_is'] = 'Quick_GO'  # Hallmark identified by GO ID

# Save processed Human Gene→BiologicalProcess (Epigenetic) edges to CSV
Human.to_csv(f'{OUT_PATH}ageanno_processed_mo/AgeAnnoMO_Human_Gene_BiologicalProcess.csv', index=None)


# ═════════════════════════════════════════════════════════════════
# MOUSE SUBSET: Gene → Epigenetic Alterations (BiologicalProcess)
# Same pipeline as Human, with mouse-specific adjustments:
# - Gene symbols capitalized (mouse convention: only first letter uppercase)
# - Uses mouse-specific NCBI symbol→description dictionary
# ═════════════════════════════════════════════════════════════════

# Filter to mouse rows only; .copy() prevents SettingWithCopyWarning
Mouse = Epigenetic_alteration[Epigenetic_alteration['Species'] == 'Mouse'].copy()
Mouse

# Replace raw methylation type with GO ID for "Epigenetic alteration" hallmark
Mouse['Tail']             = 'GO:1111113'
Mouse['Tail_detail_name'] = 'Epigenetic Alterations'

# Capitalize gene symbol to match mouse NCBI convention (e.g. Trp53, not TRP53)
Mouse['Head'] = Mouse['Head'].astype(str).str.capitalize()

# Update relation label to standard KG edge type
Mouse['Relation'] = 'Gene_BiologicalProcess'

# Map mouse gene symbol → NCBI gene description using mouse-specific dict
Mouse['Head_detail_name'] = Mouse['Head'].map(NCBI_Mouse_SYM_Descrip_dict)
Mouse

# Drop rows where gene symbol had no NCBI match in mouse dictionary
Mouse = Mouse[~Mouse['Head_detail_name'].isna()]
Mouse

# Set ID namespace labels
Mouse['Head_id_is'] = 'NCBI_ID'   # Gene identified by NCBI Gene ID
Mouse['Tail_id_is'] = 'Quick_GO'  # Hallmark identified by GO ID
Mouse

# Save processed Mouse Gene→BiologicalProcess (Epigenetic) edges to CSV
Mouse.to_csv(f'{OUT_PATH}ageanno_processed_mo/AgeAnnoMO_Mouse_Gene_BiologicalProcess.csv', index=None)

{'Human', 'Mouse'}
{'hypermethylation', 'hypomethylation'}


# 9

In [145]:
# ─────────────────────────────────────────────────────────────────
# LOAD: Age-Correlated Proteins (AgeAnnoMO - Loss of Proteostasis)
# Source: Age-correlated protein.xlsx — proteins whose abundance
# changes with age, mapped to the tissues they were measured in
# ─────────────────────────────────────────────────────────────────
Loss_of_Pr_1 = pd.read_excel(f'{BASE_PATH}ageannomo/AgeAnnoMO-main/Loss of proteostasis/Age-correlated protein.xlsx'
)
Loss_of_Pr_1


# ─────────────────────────────────────────────────────────────────
# RENAME COLUMNS to match KG schema
# Head = UniProt entry (protein), Tail = tissue, Species = organism
# ─────────────────────────────────────────────────────────────────
Loss_of_Pr_1 = Loss_of_Pr_1.rename(columns={
    'Uniprot entry': 'Head',
    'Tissue':        'Tail',
    'Animal':        'Species'
})

# Inspect which organisms are present
print(set(Loss_of_Pr_1['Species']))


# ─────────────────────────────────────────────────────────────────
# ADD METADATA COLUMNS for KG schema
# Relation: Protein expressed/detected in a Tissue during aging
# NOTE: Relation label is 'Gene_Tissue' but Head is a Protein (UniProt);
# consider updating to 'Protein_Tissue' for consistency
# ─────────────────────────────────────────────────────────────────
Loss_of_Pr_1['Head_type']       = 'Protein'
Loss_of_Pr_1['Tail_type']       = 'Tissue'
Loss_of_Pr_1['Relation']        = 'Gene_Tissue'       # NOTE: should be 'Protein_Tissue'
Loss_of_Pr_1['Relation_Source'] = 'AgeAnnoMO'
Loss_of_Pr_1['Head_id_is']      = 'Uniprot'           # Head identified by UniProt accession
Loss_of_Pr_1['Tail_id_is']      = 'BTO_ID'            # Tail identified by BRENDA Tissue Ontology ID


# ─────────────────────────────────────────────────────────────────
# REORDER columns to match KG output schema and drop duplicates
# ─────────────────────────────────────────────────────────────────
desired_order = [
    'Head', 'Relation', 'Tail',
    'Head_type', 'Tail_type',
    'Head_id_is', 'Tail_id_is',
    'Relation_Source', 'Species'
]
Loss_of_Pr_1 = Loss_of_Pr_1[desired_order]
Loss_of_Pr_1 = Loss_of_Pr_1.drop_duplicates()
Loss_of_Pr_1

# Inspect species distribution after deduplication
Loss_of_Pr_1['Species'].value_counts()


# ─────────────────────────────────────────────────────────────────
# NORMALIZE Tissue names for BTO dictionary lookup
# BTO dictionary expects clean lowercase tissue names without
# the word 'proteome' and uses 'blood plasma' instead of 'plasma'
# ─────────────────────────────────────────────────────────────────

# Lowercase all tissue names for consistent matching
Loss_of_Pr_1['Tail'] = Loss_of_Pr_1['Tail'].astype(str).str.lower()

# Remove ' proteome' suffix (e.g. 'liver proteome' → 'liver')
Loss_of_Pr_1['Tail'] = Loss_of_Pr_1['Tail'].str.replace(' proteome', '', regex=False)

# Standardize 'plasma' → 'blood plasma' to match BTO terminology
Loss_of_Pr_1['Tail'] = Loss_of_Pr_1['Tail'].str.replace('plasma', 'blood plasma', regex=False)

# Map normalized tissue name → BTO ontology ID
Loss_of_Pr_1['Tail_detail_name'] = Loss_of_Pr_1['Tail'].map(BTO_dict)

# Drop rows where tissue name had no BTO match
Loss_of_Pr_1 = Loss_of_Pr_1[~Loss_of_Pr_1['Tail_detail_name'].isna()]

# Swap: Tail ← BTO ID, Tail_detail_name ← original tissue name
# After swap: Tail = BTO ID (used in KG), Tail_detail_name = human-readable tissue name
Loss_of_Pr_1[['Tail', 'Tail_detail_name']] = Loss_of_Pr_1[['Tail_detail_name', 'Tail']]
Loss_of_Pr_1


# ═════════════════════════════════════════════════════════════════
# HUMAN SUBSET: Protein → Tissue edges for Homo sapiens
# ═════════════════════════════════════════════════════════════════

# Filter to human rows; .copy() prevents SettingWithCopyWarning
Human = Loss_of_Pr_1[Loss_of_Pr_1['Species'] == 'Human'].copy()

# Map UniProt accession → protein recommended name for readability
Human['Head_detail_name'] = Human['Head'].map(Uniprot_Recname_dict)

# Standardize species label to full scientific name
Human['Species'] = 'Homo sapiens'
Human

# Drop rows where UniProt accession had no name match
Human = Human[~Human['Head_detail_name'].isna()]

# Save processed Human Protein→Tissue edges to CSV
Human.to_csv(f'{OUT_PATH}ageanno_processed_mo/AgeAnnoMO_Human_Protein_Tissue.csv', index=None)
Human


# ═════════════════════════════════════════════════════════════════
# C. ELEGANS SUBSET: Protein → Tissue edges for Caenorhabditis elegans
# ═════════════════════════════════════════════════════════════════

# Filter to C. elegans rows; .copy() prevents SettingWithCopyWarning
Cele = Loss_of_Pr_1[Loss_of_Pr_1['Species'] == 'C. elegans'].copy()

# Map UniProt accession → protein recommended name
Cele['Head_detail_name'] = Cele['Head'].map(Uniprot_Recname_dict)

# Standardize species label to full scientific name
Cele['Species'] = 'Caenorhabditis elegans'
Cele

# Drop rows where UniProt accession had no name match
Cele = Cele[~Cele['Head_detail_name'].isna()]
Cele

# Save processed C. elegans Protein→Tissue edges to CSV
Cele.to_csv(f'{OUT_PATH}ageanno_processed_mo/AgeAnnoMO_Celegans_Protein_Tissue.csv', index=None)


# ═════════════════════════════════════════════════════════════════
# DROSOPHILA SUBSET: Protein → Tissue edges for Drosophila melanogaster
# ═════════════════════════════════════════════════════════════════

# Filter to Drosophila rows; .copy() prevents SettingWithCopyWarning
Droso = Loss_of_Pr_1[Loss_of_Pr_1['Species'] == 'Drosophila melanogaster'].copy()

# Map UniProt accession → protein recommended name
Droso['Head_detail_name'] = Droso['Head'].map(Uniprot_Recname_dict)

# Species label already correct — no rename needed for Drosophila
Droso['Species'] = 'Drosophila melanogaster'
Droso

# Drop rows where UniProt accession had no name match
Droso = Droso[~Droso['Head_detail_name'].isna()]

# Save processed Drosophila Protein→Tissue edges to CSV
Droso.to_csv(f'{OUT_PATH}ageanno_processed_mo/AgeAnnoMO_Droso_Protein_Tissue.csv', index=None)
Droso

{'Human', 'Drosophila melanogaster', 'C. elegans'}


,Head,Relation,Tail,Head_type,Tail_type,Head_id_is,Tail_id_is,Relation_Source,Species,Tail_detail_name,Head_detail_name
2370,P41572,Gene_Tissue,BTO:0000142,Protein,Tissue,Uniprot,BTO_ID,AgeAnnoMO,Drosophila melanogaster,brain,"6-phosphogluconate dehydrogenase, decarboxylating"
2448,Q9VSA3,Gene_Tissue,BTO:0000142,Protein,Tissue,Uniprot,BTO_ID,AgeAnnoMO,Drosophila melanogaster,brain,"Medium-chain specific acyl-CoA dehydrogenase, ..."
2449,Q9VLJ6,Gene_Tissue,BTO:0000142,Protein,Tissue,Uniprot,BTO_ID,AgeAnnoMO,Drosophila melanogaster,brain,Angiotensin-converting enzyme-related protein
2450,Q7KML2,Gene_Tissue,BTO:0000142,Protein,Tissue,Uniprot,BTO_ID,AgeAnnoMO,Drosophila melanogaster,brain,Acyl-coenzyme A oxidase 1 {ECO:0000305}
2451,P02574,Gene_Tissue,BTO:0000142,Protein,Tissue,Uniprot,BTO_ID,AgeAnnoMO,Drosophila melanogaster,brain,"Actin, larval muscle"
...,...,...,...,...,...,...,...,...,...,...,...
2954,Q9VTF9,Gene_Tissue,BTO:0000142,Protein,Tissue,Uniprot,BTO_ID,AgeAnnoMO,Drosophila melanogaster,brain,Ubiquitin fusion degradation protein 1 homolog...
2955,Q9W4P5,Gene_Tissue,BTO:0000142,Protein,Tissue,Uniprot,BTO_ID,AgeAnnoMO,Drosophila melanogaster,brain,V-type proton ATPase subunit d 1
2956,Q9V7D2,Gene_Tissue,BTO:0000142,Protein,Tissue,Uniprot,BTO_ID,AgeAnnoMO,Drosophila melanogaster,brain,V-type proton ATPase subunit D 1
2957,P54611,Gene_Tissue,BTO:0000142,Protein,Tissue,Uniprot,BTO_ID,AgeAnnoMO,Drosophila melanogaster,brain,V-type proton ATPase subunit E
